In [4]:
import numpy as np
from PIL import Image

In [3]:
# matriz de la transformada discreta del coseno

U = np.zeros((8,8))
for i in range(8):
    for j in range(8):
        if i == 0:
            U[i,j] = 1/np.sqrt(8) # sqrt(2)/4
        else:
            U[i,j] = np.sqrt(2/8)*np.cos(((2*j+1)*i*np.pi)/(2*8))

In [21]:
# matriz de normalización

Z = np.array([[16, 11, 10, 16, 24, 40, 51, 61],
              [12, 12, 14, 19, 26, 58, 60, 55],
              [14, 13, 16, 24, 40, 57, 69, 56],
              [14, 17, 22, 29, 51, 87, 80, 62],
              [18, 22, 37, 56, 68, 108, 103, 77],
              [24, 35, 55, 64, 81, 104, 113, 92],
              [49, 64, 78, 87, 103, 121, 120, 101],
              [72, 92, 95, 98, 112, 100, 103, 99]])


              

In [15]:
# Importamos la imagen y la convetimos a un array de numpy
img = Image.open('gato.jpeg')
gray_img = img.convert('L')
gray_img_array = np.array(gray_img)

# Recortamos la imagen para que sea de tamaño 8n x 8m
gray_img_array = gray_img_array[:gray_img_array.shape[0]//8*8,:gray_img_array.shape[1]//8*8]
print(gray_img_array.shape)

(184, 264)


In [17]:
# centramos la intensidad de la imagen
gray_img_array = gray_img_array - 128

In [18]:
# dividimos la imagen en bloques de 8x8
blocks = []
for i in range(0,gray_img_array.shape[0],8):
    for j in range(0,gray_img_array.shape[1],8):
        blocks.append(gray_img_array[i:i+8,j:j+8])

In [19]:
# calculamos la transformada discreta del coseno para cada bloque
# C = U * A * U^T
dct_blocks = []
for block in blocks:
    dct_blocks.append(U.dot(block).dot(U.T))

In [22]:
# normalizamos usando Z y redondeamos
quantized_blocks = []
for block in dct_blocks:
    quantized_blocks.append(np.round(block/Z))

In [32]:
# función para convertir un bloque en un vector siguiendo el zig-zag
def block_to_vector(block):
    vector = []
    row, col = 0, 0
    rows, cols = block.shape
    direction = 1 # 1: hacia arriba, -1: hacia abajo

    for _ in range(rows * cols):
        vector.append(block[row, col])
        
        # cambiamos dirección si llegamos a un borde
        if direction == 1:
            if row == 0 or col == cols - 1:
                direction = -1
                if col < cols - 1:
                    col += 1
                else:
                    row += 1
            else:
                row -= 1
                col += 1
        else:
            if col == 0 or row == rows - 1:
                direction = 1
                if row < rows - 1:
                    row += 1
                else:
                    col += 1
            else:
                row += 1
                col -= 1
    return np.array(vector)


In [33]:
for i in range(len(quantized_blocks)):
    quantized_blocks[i] = block_to_vector(quantized_blocks[i])

#concatenamos los vectores
quantized_blocks = np.array(quantized_blocks)


In [34]:
print(quantized_blocks.shape)

(759, 64)


In [ ]:
# creamos árbol de huffman

from heapq import heappush, heappop, heapify
from collections import defaultdict
import os

class Node:
    def __init__(self, freq, symbol, left=None, right=None):
        self.freq = freq
        self.symbol = symbol
        self.left = left
        self.right = right
        self.huff = ''

    def __lt__(self, other):
        return self.freq < other.freq
    
    def __eq__(self, other):
        return self.freq == other.freq
    
    def __gt__(self, other):
        return self.freq > other.freq
    
    def __le__(self, other):
        return self.freq <= other.freq
    
    def __ge__(self, other):
        return self.freq >= other.freq
    
    def __ne__(self, other):
        return self.freq != other.freq
    
    def __repr__(self):
        return f'Node({self.freq}, {self.symbol}, {self.left}, {self.right})'

def huffman_tree(data):
    heap = []
    for symbol, freq in data.items():
        heappush(heap, Node(freq, symbol))
    
    while len(heap) > 1:
        left = heappop(heap)
        right = heappop(heap)
        heappush(heap, Node(left.freq + right.freq, None, left, right))
    
    return heappop(heap)

data = defaultdict(int)
for block in quantized_blocks:
    for symbol in block:
        data[symbol] += 1

